In [18]:
import torch
from torch import nn, optim
from torch.distributions import Normal
from torch.nn.functional import mse_loss
from tqdm import trange

In [19]:
state_dim = 4
message_dim = 32
action_dim = state_dim
hidden_dim = 64
num_layers = 3

def reward_fn(state, action):
    return 100 - ((state - action) ** 2).sum(dim=-1, keepdim=True)

In [20]:
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim, num_layers):
        super(MLP, self).__init__()
        layers = []
        for i in range(num_layers):
            layers.append(nn.Linear(input_dim if i == 0 else hidden_dim, hidden_dim))
            layers.append(nn.LayerNorm(hidden_dim))
            layers.append(nn.LeakyReLU())
        layers.append(nn.Linear(hidden_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, *inputs):
        return self.net(torch.cat(inputs, dim=-1))

    def get_normal_policy(self, *inputs):
        mean, log_std = self.forward(*inputs).chunk(2, dim=-1)
        std = log_std.clamp(-20, 2).exp()
        return Normal(mean, std)

In [21]:
num_episodes = 50000
batch_size = 64
epsilon = 0.1
k = 4
alpha = 0.01
error_factor = 5.0

In [22]:
# Actors — decentralized, only see what flows through the chain
message1_actor = MLP(state_dim,              2 * message_dim, hidden_dim, num_layers)
message2_actor = MLP(state_dim + message_dim, 2 * message_dim, hidden_dim, num_layers)
final_actor    = MLP(message_dim,            2 * action_dim,  hidden_dim, num_layers)

# Critics — centralized, see state during training
message1_critic = MLP(state_dim,                       1, hidden_dim, num_layers)
message2_critic = MLP(state_dim + message_dim,         1, hidden_dim, num_layers)
final_critic    = MLP(2 * message_dim,                 1, hidden_dim, num_layers)

message1_actor_opt  = optim.Adam(message1_actor.parameters(),  lr=3e-4)
message2_actor_opt  = optim.Adam(message2_actor.parameters(),  lr=3e-4)
final_actor_opt     = optim.Adam(final_actor.parameters(),     lr=3e-4)
message1_critic_opt = optim.Adam(message1_critic.parameters(), lr=3e-4)
message2_critic_opt = optim.Adam(message2_critic.parameters(), lr=3e-4)
final_critic_opt    = optim.Adam(final_critic.parameters(),    lr=3e-4)

In [23]:
# I AM THE LIVING MAN
for episode in trange(num_episodes):
    # Collect rollout samples
    with torch.no_grad():
        state = torch.randn(k * batch_size, state_dim)

        message1_policy = message1_actor.get_normal_policy(state)
        message1 = message1_policy.sample()
        message1_log_prob = message1_policy.log_prob(message1).sum(dim=-1, keepdim=True)

        message2_policy = message2_actor.get_normal_policy(torch.zeros_like(state), message1)
        message2 = message2_policy.sample()
        message2_log_prob = message2_policy.log_prob(message2).sum(dim=-1, keepdim=True)

        final_policy = final_actor.get_normal_policy(message2)
        action = final_policy.sample()
        action_log_prob = final_policy.log_prob(action).sum(dim=-1, keepdim=True)

        final_reward = reward_fn(state, action)

    # Update critics and actors
    for i in range(k):
        mb_state           = state[i * batch_size:(i + 1) * batch_size]
        mb_message1        = message1[i * batch_size:(i + 1) * batch_size]
        mb_message1_lp     = message1_log_prob[i * batch_size:(i + 1) * batch_size]
        mb_message2        = message2[i * batch_size:(i + 1) * batch_size]
        mb_message2_lp     = message2_log_prob[i * batch_size:(i + 1) * batch_size]
        mb_action          = action[i * batch_size:(i + 1) * batch_size]
        mb_action_lp       = action_log_prob[i * batch_size:(i + 1) * batch_size]
        mb_final_reward    = final_reward[i * batch_size:(i + 1) * batch_size]

        # -- Critics --
        message1_critic_opt.zero_grad()
        message1_critic_value = message1_critic(mb_state)
        message2_critic_value = message2_critic(mb_state, mb_message1)
        final_critic_value    = final_critic(mb_message1, mb_message2)

        # target = next critic value + error_factor * |r - next critic value|
        m1_target = (message2_critic_value + error_factor * (mb_final_reward - message2_critic_value).abs()).detach()
        m2_target = (final_critic_value    + error_factor * (mb_final_reward - final_critic_value).abs()).detach()

        message1_critic_loss = mse_loss(message1_critic_value, m1_target)
        message1_critic_loss.backward()
        message1_critic_opt.step()

        message2_critic_opt.zero_grad()
        message2_critic_loss = mse_loss(message2_critic_value, m2_target)
        message2_critic_loss.backward()
        message2_critic_opt.step()

        final_critic_opt.zero_grad()
        final_critic_loss = mse_loss(final_critic_value, mb_final_reward)
        final_critic_loss.backward()
        final_critic_opt.step()

        # -- Marginal advantages --
        with torch.no_grad():
            v0 = message1_critic(mb_state)
            v1 = message2_critic(mb_state, mb_message1)
            v2 = final_critic(mb_message1, mb_message2)

        message1_advantage = v1 - v0
        message2_advantage = v2 - v1
        final_advantage    = mb_final_reward - v2

        def normalize(adv):
            return (adv - adv.mean()) / (adv.std() + 1e-8)

        message1_advantage = normalize(message1_advantage)
        message2_advantage = normalize(message2_advantage)
        final_advantage    = normalize(final_advantage)

        # -- Actor 1 --
        message1_actor_opt.zero_grad()
        message1_policy       = message1_actor.get_normal_policy(mb_state)
        message1_new_lp       = message1_policy.log_prob(mb_message1).sum(dim=-1, keepdim=True)
        message1_ratio        = (message1_new_lp - mb_message1_lp).exp()
        message1_ratio_clip   = message1_ratio.clamp(1 - epsilon, 1 + epsilon)
        message1_ent          = message1_policy.entropy().mean()
        message1_actor_loss   = -torch.min(
            message1_ratio * message1_advantage,
            message1_ratio_clip * message1_advantage
        ).mean() - alpha * message1_ent
        message1_actor_loss.backward()
        message1_actor_opt.step()

        # -- Actor 2 --
        message2_actor_opt.zero_grad()
        message2_policy       = message2_actor.get_normal_policy(torch.zeros_like(mb_state), mb_message1)
        message2_new_lp       = message2_policy.log_prob(mb_message2).sum(dim=-1, keepdim=True)
        message2_ratio        = (message2_new_lp - mb_message2_lp).exp()
        message2_ratio_clip   = message2_ratio.clamp(1 - epsilon, 1 + epsilon)
        message2_ent          = message2_policy.entropy().mean()
        message2_actor_loss   = -torch.min(
            message2_ratio * message2_advantage,
            message2_ratio_clip * message2_advantage
        ).mean() - alpha * message2_ent
        message2_actor_loss.backward()
        message2_actor_opt.step()

        # -- Final actor --
        final_actor_opt.zero_grad()
        final_policy          = final_actor.get_normal_policy(mb_message2)
        final_new_lp          = final_policy.log_prob(mb_action).sum(dim=-1, keepdim=True)
        final_ratio           = (final_new_lp - mb_action_lp).exp()
        final_ratio_clip      = final_ratio.clamp(1 - epsilon, 1 + epsilon)
        final_ent             = final_policy.entropy().mean()
        final_actor_loss      = -torch.min(
            final_ratio * final_advantage,
            final_ratio_clip * final_advantage
        ).mean() - alpha * final_ent
        final_actor_loss.backward()
        final_actor_opt.step()

    if episode % 100 == 0:
        print(
            f"{episode} | "
            f"fcl: {final_critic_loss.item():.4f} | "
            f"m2cl: {message2_critic_loss.item():.4f} | "
            f"m1cl: {message1_critic_loss.item():.4f} | "
            f"fr: {final_reward.mean().item():.4f}"
        )

  0%|          | 3/50000 [00:00<1:18:47, 10.58it/s]

0 | fcl: 8057.6318 | m2cl: 201619.9531 | m1cl: 202372.6562 | fr: 88.1583


  0%|          | 103/50000 [00:09<1:12:08, 11.53it/s]

100 | fcl: 7717.5566 | m2cl: 192571.2969 | m1cl: 190958.7500 | fr: 95.8008


  0%|          | 203/50000 [00:17<1:13:24, 11.31it/s]

200 | fcl: 6398.4121 | m2cl: 159822.4375 | m1cl: 159174.0469 | fr: 96.0187


... (991 outputs left out)

49800 | fcl: 5.2952 | m2cl: 31.9371 | m1cl: 304.7585 | fr: 95.6328


100%|█████████▉| 49902/50000 [11:06:34<00:11,  8.47it/s]

49900 | fcl: 5.4119 | m2cl: 29.5878 | m1cl: 192.0796 | fr: 96.0152


100%|██████████| 50000/50000 [11:06:46<00:00,  1.25it/s]
